In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt

In [2]:
# Load dataset
path="../../data/simon-data/properties-1-cleaned.csv"
properties=pd.read_csv(path,index_col=0)
properties.reset_index()
properties.isna().mean()

URL                       0.000000
Zip code                  0.000000
City                      0.000000
Type of property          0.000000
Subtype of property       0.000000
Price                     0.000000
Type of sale              0.000000
Number of rooms           0.016056
Living Area               0.000000
Fully equipped kitchen    0.640534
Furnished                 0.000000
Open fire                 0.000000
Terrace                   0.000000
Garden                    0.000000
Garden area               0.310396
Surface of the land       0.529778
Number of facades         0.206111
Swimming pool             0.000000
State of the building     0.202097
dtype: float64

In [3]:
# Extrapolation
## Get columns with missing values
miss_col=properties.isna().mean()[properties.isna().mean()>0].index

In [4]:
# Get stats on missing values
print(f'Means:\n{properties[miss_col].mean()}\n')
print(f'Medians:\n{properties[miss_col].median()}\n')
print(f'Std:\n{properties[miss_col].std()}')

Means:
Number of rooms              2.930064
Fully equipped kitchen       2.224932
Garden area                369.398432
Surface of the land       1187.629094
Number of facades            2.739346
State of the building        1.604928
dtype: float64

Medians:
Number of rooms             3.0
Fully equipped kitchen      2.0
Garden area                 0.0
Surface of the land       488.0
Number of facades           2.0
State of the building       1.0
dtype: float64

Std:
Number of rooms              1.623958
Fully equipped kitchen       0.706057
Garden area               7779.768459
Surface of the land       3514.566401
Number of facades            0.839405
State of the building        1.387969
dtype: float64


In [5]:
# Remove outliers (3*IQR = soft removing >< 1.5*IQR)
## Get quartiles
Q1=properties[miss_col].quantile(0.25)
print(Q1)
Q3=properties[miss_col].quantile(0.75)
print(Q3)
IQR=Q3-Q1
print(IQR)

## Get limits
low_limit=(Q1-3*IQR).clip(lower=0)
print(low_limit)
high_limit=Q3+3*IQR
print(high_limit)

### Find limit value for 'Garden area' for prop with garden
prop_with_garden=(properties[properties['Garden area']>0]==True).index
prop_with_garden=properties.loc[prop_with_garden]
gardenQ1=prop_with_garden['Garden area'].quantile(0.25)
gardenQ3=prop_with_garden['Garden area'].quantile(0.75)
gardenIQR=gardenQ3-gardenQ1
garden_high_limit=gardenQ3+3*gardenIQR

### Change limit in high_limit
high_limit['Garden area']=garden_high_limit
high_limit

## Exclude outliers
### Create SubDataSet with replaced NaN values
sub_excl_manip=properties
sub_excl_manip=sub_excl_manip.replace(np.nan,0)
mask=sub_excl_manip[miss_col].apply(lambda x: x.between(low_limit[x.name],high_limit[x.name])).all(axis=1)

### Retrieve Dataset without outliers
properties=properties[mask]

Number of rooms             2.00
Fully equipped kitchen      2.00
Garden area                 0.00
Surface of the land       221.75
Number of facades           2.00
State of the building       1.00
Name: 0.25, dtype: float64
Number of rooms              4.0
Fully equipped kitchen       3.0
Garden area                 90.0
Surface of the land       1027.5
Number of facades            4.0
State of the building        2.0
Name: 0.75, dtype: float64
Number of rooms             2.00
Fully equipped kitchen      1.00
Garden area                90.00
Surface of the land       805.75
Number of facades           2.00
State of the building       1.00
dtype: float64
Number of rooms           0.0
Fully equipped kitchen    0.0
Garden area               0.0
Surface of the land       0.0
Number of facades         0.0
State of the building     0.0
dtype: float64
Number of rooms             10.00
Fully equipped kitchen       6.00
Garden area                360.00
Surface of the land       3444.75
Number

In [6]:
# Extrapolation
## Check means/median by column
properties_means=properties[miss_col].mean()
properties[miss_col].median

## Change Garden area by subset for gardened prop
sub_garden_area=sub_excl_manip['Garden area'].mean()
properties_means['Garden area']=sub_garden_area

## Round int values (rooms, kitch)
properties_means=properties_means.round()
properties_means

## replace Nan by means
properties[miss_col]=properties[miss_col].fillna(properties_means)


In [7]:
# Save the new dataset
properties.to_csv('../../data/simon-data/properties-2-extrapolated.csv')

In [8]:
properties.isna().mean()

URL                       0.0
Zip code                  0.0
City                      0.0
Type of property          0.0
Subtype of property       0.0
Price                     0.0
Type of sale              0.0
Number of rooms           0.0
Living Area               0.0
Fully equipped kitchen    0.0
Furnished                 0.0
Open fire                 0.0
Terrace                   0.0
Garden                    0.0
Garden area               0.0
Surface of the land       0.0
Number of facades         0.0
Swimming pool             0.0
State of the building     0.0
dtype: float64

In [9]:
properties

,URL,Zip code,City,Type of property,Subtype of property,Price,Type of sale,Number of rooms,Living Area,Fully equipped kitchen,Furnished,Open fire,Terrace,Garden,Garden area,Surface of the land,Number of facades,Swimming pool,State of the building
1,https://immovlan.be/en/detail/residence/for-sa...,2200,Herentals,House,residence,369000,for sale,3.0,378,2.0,0,0,1,1,255.0,300.0,2.0,0,0.0
2,https://immovlan.be/en/detail/apartment/for-sa...,1190,Vorst,Apartment,apartment,385000,for sale,2.0,100,2.0,0,0,1,0,0.0,649.0,3.0,0,2.0
3,https://immovlan.be/en/detail/residence/for-sa...,7100,La Louvière,House,residence,135000,for sale,3.0,125,1.0,0,0,0,1,255.0,649.0,2.0,0,2.0
5,https://immovlan.be/en/detail/residence/for-sa...,2160,Wommelgem,House,residence,284000,for sale,3.0,180,2.0,0,0,0,1,240.0,295.0,2.0,0,0.0
6,https://immovlan.be/en/detail/residence/for-sa...,6660,Houffalize,House,residence,299000,for sale,4.0,124,3.0,0,0,1,1,255.0,441.0,3.0,0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13243,https://immovlan.be/en/detail/apartment/for-sa...,1090,Jette,Apartment,apartment,410000,for sale,4.0,197,2.0,0,0,1,0,0.0,649.0,3.0,0,0.0
13244,https://immovlan.be/en/detail/residence/for-sa...,9190,Stekene,House,residence,450000,for sale,3.0,170,2.0,0,0,0,1,255.0,300.0,3.0,0,4.0
13245,https://immovlan.be/en/detail/apartment/for-sa...,1030,Schaarbeek,Apartment,apartment,229000,for sale,1.0,72,1.0,0,0,0,0,0.0,649.0,3.0,0,1.0
13246,https://immovlan.be/en/detail/residence/for-sa...,6470,Sautin,House,residence,199000,for sale,3.0,134,2.0,0,1,1,1,255.0,1497.0,3.0,0,2.0
